In [1]:
import pandas as pd
import lseg.data as ld

from pathlib import Path
from datetime import date

In [2]:
pd.set_option('future.no_silent_downcasting', True)
pd.set_option('display.max_rows', None)

In [3]:
ld.open_session()

<lseg.data.session.Definition object at 0x10f131e80 {name='workspace'}>

# Fecthing Data for Equity Options


The option chains follow the given naming logic: `0#<RIC-root>*.<exchange>`, where `RIC-root` is the company abbreviation (i.e. `MUVGn` for Münchener Rückversicherung) and `exchange` identifies the place of the stock exchange at hand (i.e. `DE` for Germany).

In [4]:
today = date.today()

START_DATE = today.strftime("%Y-%m-%d")
AS_OF_DATE = pd.to_datetime(START_DATE).normalize()

DATA_DIR = Path('./Option Data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

MAX_MATURITY_YEARS = 1.5
UNDERLYING_PRICE_LOOKBACK_DAYS = 10

OPTION_FIELDS = ['PUTCALLIND', 'PUT_CALL', 'EXPIR_DATE', 'STRIKE_PRC', 'CF_CLOSE', 'VEGA']
UNDERLYING_FIELD = 'TRDPRC_1' #'TRDPRC_1', OFF_CLOSE

tickers = [
    'ASML.AS',
    'LVMH.PA',
    'SIEGn.DE',
    'OREP.PA',
    'PRX.AS',
    'HRMS.PA',
    'SAPG.DE',
    'TTEF.PA',
    'ITX.MC',
    'SCHN.PA',
]

RICs = [
    '0#ASME*.EX',   # ASML, Eurex
    '0#MOHE*.EX',   # LVMH, Eurex
    '0#SIEE*.EX',   # Siemens, Eurex
    '0#LORE*.EX',   # L'Oréal, Eurex
    '0#PRXE*.EX',     # Prosus, Eurex
    '0#HMIE*.EX',   # Hermès, Eurex
    '0#SAPE*.EX',   # SAP, Eurex
    '0#TOTE*.EX',   # TotalEnergies, Eurex
    '0#IXDE*.EX',    # Inditex, Eurex
    '0#SNDE*.EX',   # Schneider Electric, Eurex
]

In [5]:
output_file_list = []
call_price_dfs = []

for i, RIC in enumerate(RICs):
    print(tickers[i], RIC)
    df = ld.get_data(universe=RIC, fields=OPTION_FIELDS)
    df = df.drop(index=df.index[0])

    call_price_df = df[df['PUTCALLIND'].str.strip().eq('CALL')].copy()
    call_price_df = call_price_df.rename(columns={'CF_CLOSE': 'price', 'STRIKE_PRC': 'strike', 'EXPIR_DATE': 'expiry_date', 'PUTCALLIND': 'option_type', 'VEGA': 'vega'})
    call_price_df['time_to_maturity_years'] = (pd.to_datetime(call_price_df['expiry_date'])-AS_OF_DATE)/pd.Timedelta(days=365)
    call_price_df = call_price_df[call_price_df['time_to_maturity_years'] <= MAX_MATURITY_YEARS]

    start_lookup_date = (AS_OF_DATE - pd.Timedelta(days=UNDERLYING_PRICE_LOOKBACK_DAYS)).strftime('%Y-%m-%d')
    end_lookup_date = AS_OF_DATE.strftime('%Y-%m-%d')
    
    close_price_stock = ld.get_history(universe=tickers[i], interval='1min', count=1, fields=UNDERLYING_FIELD)
    close_price_stock = float(pd.to_numeric(close_price_stock[UNDERLYING_FIELD], errors='coerce').dropna().iloc[-1])

    call_price_df['S0'] = round(close_price_stock, 5)

    output_file = DATA_DIR/f'{RIC}.xlsx'
    call_price_df.to_excel(output_file, index=False)

    call_price_dfs.append(call_price_df)
    output_file_list.append(str(output_file))

ASML.AS 0#ASME*.EX
LVMH.PA 0#MOHE*.EX
SIEGn.DE 0#SIEE*.EX
OREP.PA 0#LORE*.EX
PRX.AS 0#PRXE*.EX
HRMS.PA 0#HMIE*.EX
SAPG.DE 0#SAPE*.EX
TTEF.PA 0#TOTE*.EX
ITX.MC 0#IXDE*.EX
SCHN.PA 0#SNDE*.EX


In [6]:
for i, df in enumerate(call_price_dfs):
    print('=' * 80)
    print(f'Overview for dataframe {i}')
    print('=' * 80)

    valuation_date = df['valuation_date'].iloc[0] if 'valuation_date' in df.columns else AS_OF_DATE.date()
    s0 = df['S0'].iloc[0]

    print(f'Underlying:      {tickers[i]}')
    print(f'Valuation date:  {valuation_date}')
    print(f'S0:              {s0}')
    print(f'Total samples:   {len(df)}')
    print()

    overview = df.groupby('expiry_date').size().reset_index(name='samples')
    overview['time_to_maturity_days'] = (pd.to_datetime(overview['expiry_date']) - pd.to_datetime(valuation_date)).dt.days
    overview = overview[['expiry_date', 'time_to_maturity_days', 'samples']]

    print(overview.to_string(index=False))
    print()

Overview for dataframe 0
Underlying:      ASML.AS
Valuation date:  2026-05-14
S0:              1367.0
Total samples:   307

expiry_date  time_to_maturity_days  samples
 2026-05-15                      1       52
 2026-06-19                     36       68
 2026-07-17                     64       43
 2026-09-18                    127       41
 2026-12-18                    218       42
 2027-03-19                    309       32
 2027-06-18                    400       29

Overview for dataframe 1
Underlying:      LVMH.PA
Valuation date:  2026-05-14
S0:              460.85
Total samples:   249

expiry_date  time_to_maturity_days  samples
 2026-05-15                      1       40
 2026-06-19                     36       57
 2026-07-17                     64       33
 2026-09-18                    127       33
 2026-12-18                    218       37
 2027-03-19                    309       28
 2027-06-18                    400       21

Overview for dataframe 2
Underlying:      SIEG

# Fetching Stocks data

In [7]:
DATA_DIR = Path('./Stock Data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

In [8]:
last_year = today.replace(year=today.year - 1)
LAST_YEAR = last_year.strftime('%Y-%m-%d')
END_DATE = START_DATE

ticker_dfs = []
for ticker in tickers:
    print(ticker)
    ticker_df = ld.get_history(universe=ticker, interval='daily', start=LAST_YEAR, end=START_DATE, fields='OFF_CLOSE')
    filename_df = DATA_DIR/f'{ticker}.xlsx'

    ticker_df.to_excel(filename_df)
    ticker_dfs.append(ticker_df)

ASML.AS
LVMH.PA
SIEGn.DE
OREP.PA
PRX.AS
HRMS.PA
SAPG.DE
TTEF.PA
ITX.MC
SCHN.PA


In [9]:
ld.close_session()